# SEARCH FOR S2 IMAGES

In this notebook we will look for S2 Images in the area of interest.

In [1]:
%load_ext autoreload
%autoreload 2

import earthaccess

from swot_toolkit.data_access import auth_earthaccess, swot_results_to_df
from swot_toolkit.kml import read_kml_geometry
from swot_toolkit.planetary import search_s2, s2_results_to_df


In [2]:
auth_earthaccess()

## Open the AOI

In [3]:
# Read geometry from a KML file
AOIs = read_kml_geometry("/data/swot/AOIs/Curua-Una.kml")
# Define the date range and Sentinel-2 tile
START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

if AOIs is not None:
    AOI = AOIs[0]

## Search for SWOT images

In [4]:
# Search for products within the AOI and Time frame
results = earthaccess.search_data(
    short_name="SWOT_L2_HR_PIXC_2.0",
    temporal=(START_DATE, END_DATE),
    bounding_box=AOI.bounds,
)

swot_df = swot_results_to_df(results, drop_duplicates=True)

swot_df.head(3)

,cycle_id,pass_id,tile_id,date_str,tile_name,vers,item,datetime,date
short_id,,,,,,,,,
SWOT_L2_HR_PIXC_009_033_149R_20240105T072529,009,033,149R,20240105T072529,033_149R,PIC0,"{'meta': {'concept-type': 'granule', 'concept-...",2024-01-05 07:25:29,2024-01-05
SWOT_L2_HR_PIXC_009_033_149L_20240105T072529,009,033,149L,20240105T072529,033_149L,PIC0,"{'meta': {'concept-type': 'granule', 'concept-...",2024-01-05 07:25:29,2024-01-05
SWOT_L2_HR_PIXC_009_033_150R_20240105T072539,009,033,150R,20240105T072539,033_150R,PIC0,"{'meta': {'concept-type': 'granule', 'concept-...",2024-01-05 07:25:39,2024-01-05


## Search Sentinel-2 on Planetary Computer

In [5]:
date_range = (START_DATE + "/" + END_DATE)
date_range

'2024-01-01/2025-12-31'

In [6]:
s2_results = search_s2(aoi=AOI, date_range=date_range, s2_tile="21MYS")
len(s2_results)



116

In [7]:
s2_df = s2_results_to_df(s2_results)
s2_df.head(3)

,datetime,tile,item
S2A_MSIL2A_20240101T135701_R067_T21MYS_20240101T191412,2024-01-01 13:57:01.024,21MYS,<Item id=S2A_MSIL2A_20240101T135701_R067_T21MY...
S2B_MSIL2A_20240106T135659_R067_T21MYS_20240106T171445,2024-01-06 13:56:59.024,21MYS,<Item id=S2B_MSIL2A_20240106T135659_R067_T21MY...
S2A_MSIL2A_20240111T135701_R067_T21MYS_20240111T191300,2024-01-11 13:57:01.024,21MYS,<Item id=S2A_MSIL2A_20240111T135701_R067_T21MY...


In [8]:
from swot_toolkit.planetary import find_closest_s2, assess_s2_clouds, match_swot_s2

In [63]:
assess_s2_clouds(
    ref_time="2024-08-02",
    s2df=s2_df,
    aoi=AOI,
    max_days=15,
)

,datetime,tile,item,delta,valid_pxls
S2B_MSIL2A_20240803T135709_R067_T21MYS_20240803T161428,2024-08-03 13:57:09.024,21MYS,<Item id=S2B_MSIL2A_20240803T135709_R067_T21MY...,1 days 13:57:09.024000,0.922366
S2A_MSIL2A_20240729T140051_R067_T21MYS_20240729T232452,2024-07-29 14:00:51.024,21MYS,<Item id=S2A_MSIL2A_20240729T140051_R067_T21MY...,-4 days +14:00:51.024000,0.912399
S2A_MSIL2A_20240808T140051_R067_T21MYS_20240808T205355,2024-08-08 14:00:51.024,21MYS,<Item id=S2A_MSIL2A_20240808T140051_R067_T21MY...,6 days 14:00:51.024000,0.994576
S2B_MSIL2A_20240724T135709_R067_T21MYS_20240724T194013,2024-07-24 13:57:09.024,21MYS,<Item id=S2B_MSIL2A_20240724T135709_R067_T21MY...,-9 days +13:57:09.024000,0.106616
S2B_MSIL2A_20240813T135709_R067_T21MYS_20240813T174653,2024-08-13 13:57:09.024,21MYS,<Item id=S2B_MSIL2A_20240813T135709_R067_T21MY...,11 days 13:57:09.024000,0.915780
S2A_MSIL2A_20240719T135701_R067_T21MYS_20240719T210440,2024-07-19 13:57:01.024,21MYS,<Item id=S2A_MSIL2A_20240719T135701_R067_T21MY...,-14 days +13:57:01.024000,0.983059


In [9]:
swot_s2 = match_swot_s2(
    swot_df=swot_df,
    s2_df=s2_df,
    aoi=AOI,
    max_days=5,
)

  0%|          | 0/158 [00:00<?, ?it/s]

KeyError: "None of ['index'] are in the columns"

In [ ]:
swot_s2

In [41]:
s2_df["datetime"] = pd.to_datetime(s2_df["datetime"]).dt.tz_localize(None)

In [42]:
s2_df["datetime"]

S2A_MSIL2A_20240101T135701_R067_T21MYS_20240101T191412   2024-01-01 13:57:01.024
S2B_MSIL2A_20240106T135659_R067_T21MYS_20240106T171445   2024-01-06 13:56:59.024
S2A_MSIL2A_20240111T135701_R067_T21MYS_20240111T191300   2024-01-11 13:57:01.024
S2B_MSIL2A_20240116T135659_R067_T21MYS_20240116T181414   2024-01-16 13:56:59.024
S2A_MSIL2A_20240121T135651_R067_T21MYS_20240121T191350   2024-01-21 13:56:51.024
                                                                   ...          
S2A_MSIL2A_20250606T140111_R067_T21MYS_20250606T185113   2025-06-06 14:01:11.024
S2B_MSIL2A_20250609T135659_R067_T21MYS_20250609T172547   2025-06-09 13:56:59.024
S2C_MSIL2A_20250614T135721_R067_T21MYS_20250614T191817   2025-06-14 13:57:21.025
S2A_MSIL2A_20250616T140101_R067_T21MYS_20250616T164918   2025-06-16 14:01:01.024
S2B_MSIL2A_20250619T135709_R067_T21MYS_20250619T174101   2025-06-19 13:57:09.024
Name: datetime, Length: 116, dtype: datetime64[ns]

In [43]:
find_closest_s2(
    ref_time=swot_df["datetime"].iloc[0],
    s2df=s2_df,
    max_days=5,
).head(3)

,datetime,tile,item,delta
S2B_MSIL2A_20240106T135659_R067_T21MYS_20240106T171445,2024-01-06 13:56:59.024,21MYS,<Item id=S2B_MSIL2A_20240106T135659_R067_T21MY...,1 days 06:31:30.024000
S2A_MSIL2A_20240101T135701_R067_T21MYS_20240101T191412,2024-01-01 13:57:01.024,21MYS,<Item id=S2A_MSIL2A_20240101T135701_R067_T21MY...,-4 days +06:31:32.024000


In [37]:
import pandas as pd
pd.to_datetime(s2_df["datetime"])

S2A_MSIL2A_20240101T135701_R067_T21MYS_20240101T191412   2024-01-01 13:57:01.024000+00:00
S2B_MSIL2A_20240106T135659_R067_T21MYS_20240106T171445   2024-01-06 13:56:59.024000+00:00
S2A_MSIL2A_20240111T135701_R067_T21MYS_20240111T191300   2024-01-11 13:57:01.024000+00:00
S2B_MSIL2A_20240116T135659_R067_T21MYS_20240116T181414   2024-01-16 13:56:59.024000+00:00
S2A_MSIL2A_20240121T135651_R067_T21MYS_20240121T191350   2024-01-21 13:56:51.024000+00:00
                                                                       ...               
S2A_MSIL2A_20250606T140111_R067_T21MYS_20250606T185113   2025-06-06 14:01:11.024000+00:00
S2B_MSIL2A_20250609T135659_R067_T21MYS_20250609T172547   2025-06-09 13:56:59.024000+00:00
S2C_MSIL2A_20250614T135721_R067_T21MYS_20250614T191817   2025-06-14 13:57:21.025000+00:00
S2A_MSIL2A_20250616T140101_R067_T21MYS_20250616T164918   2025-06-16 14:01:01.024000+00:00
S2B_MSIL2A_20250619T135709_R067_T21MYS_20250619T174101   2025-06-19 13:57:09.024000+00:00
Name: date

In [29]:
swot_df

,cycle_id,pass_id,tile_id,date_str,tile_name,vers,item,datetime,date
short_id,,,,,,,,,
SWOT_L2_HR_PIXC_009_033_149R_20240105T072529,009,033,149R,20240105T072529,033_149R,PIC0,"{'meta': {'concept-type': 'granule', 'concept-...",2024-01-05 07:25:29,2024-01-05
SWOT_L2_HR_PIXC_009_033_149L_20240105T072529,009,033,149L,20240105T072529,033_149L,PIC0,"{'meta': {'concept-type': 'granule', 'concept-...",2024-01-05 07:25:29,2024-01-05
SWOT_L2_HR_PIXC_009_033_150R_20240105T072539,009,033,150R,20240105T072539,033_150R,PIC0,"{'meta': {'concept-type': 'granule', 'concept-...",2024-01-05 07:25:39,2024-01-05
SWOT_L2_HR_PIXC_009_033_150L_20240105T072539,009,033,150L,20240105T072539,033_150L,PIC0,"{'meta': {'concept-type': 'granule', 'concept-...",2024-01-05 07:25:39,2024-01-05
SWOT_L2_HR_PIXC_009_186_159R_20240110T183829,009,186,159R,20240110T183829,186_159R,PIC0,"{'meta': {'concept-type': 'granule', 'concept-...",2024-01-10 18:38:29,2024-01-10
...,...,...,...,...,...,...,...,...,...
SWOT_L2_HR_PIXC_031_186_160R_20250413T191023,031,186,160R,20250413T191023,186_160R,PIC2,"{'meta': {'concept-type': 'granule', 'concept-...",2025-04-13 19:10:23,2025-04-13
SWOT_L2_HR_PIXC_032_033_149R_20250429T044218,032,033,149R,20250429T044218,033_149R,PIC2,"{'meta': {'concept-type': 'granule', 'concept-...",2025-04-29 04:42:18,2025-04-29
SWOT_L2_HR_PIXC_032_033_149L_20250429T044218,032,033,149L,20250429T044218,033_149L,PIC2,"{'meta': {'concept-type': 'granule', 'concept-...",2025-04-29 04:42:18,2025-04-29


In [21]:
import pystac_client

In [22]:
type(s2_df)

pystac.item_collection.ItemCollection

In [ ]:
s2_df.